# CLOY Processing

## Kailyn

This notebook is for you to work with the CLOY data folder.

We're working with `cloy_chronological.jsonl`, a collection of Reddit threads from the **CrashLandingOnYou** subreddit.

Each line of the JSONL file represents one Reddit thread, which contains:

- the original post
- a nested list of comments
- comments can contain replies
- replies can contain further replies

The main goal is to turn each complete thread into a vector of words and then calculate the cosine similarity between every pair of threads.

## Table of Contents

1. [Set up the libraries](#1)
2. [Load the JSONL data](#2)
3. [Inspect the structure of one thread](#3)
4. [Flatten the nested comments](#4)
5. [Turn each thread into text](#5)
6. [Tokenize with TextBlob](#6)
7. [Create word vectors](#7)
8. [Calculating Cosine Similarity](#8)
9. [Creating the Similarity DataFrame](#9)
10. [Find the five most similar threads](#10)
11. [Verification](#11)
12. [Stop words: repeat the process](#12)
13. [Compare the two approaches](#13)

<h2 id="1">1. Set up the libraries</h2>

(We include the basic things like json, Path, pandas, and numpy, but also TextBlob, which we recently covered.)

In [1]:
import json
from pathlib import Path
from collections import Counter
from itertools import combinations

import numpy as np
from numpy.linalg import norm
import pandas as pd

from textblob import TextBlob
from nltk.corpus import stopwords

<a id="2"></a>

## 2. Load the JSONL data

Fairly straightforward...

A JSONL file consists of multiple JSON objects, one per line, so I read in the file one line at a time and call `json.loads()` on each line.

In [2]:
DATA_PATH = Path("data/cs315-week4-data/RedditExplorer/cloy_chronological.jsonl")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}." 
    )

threads = []

with DATA_PATH.open("r", encoding="utf-8") as file:
    for line in file:
        line = line.strip()
        if line:
            threads.append(json.loads(line))

print(f"Loaded {len(threads):,} threads.")
threads[0]

Loaded 3,075 threads.


{'post_id': 'dkxewd',
 'timestamp': 1571646432,
 'author': '[deleted]',
 'subreddit': 'CrashLandingOnYou',
 'title': 'Crash Landing On You has been created',
 'selftext': '[deleted]',
 'url': 'https://www.reddit.com/r/CrashLandingOnYou/comments/dkxewd/crash_landing_on_you_has_been_created/',
 'score': 1,
 'permalink': 'https://reddit.com/r/CrashLandingOnYou/comments/dkxewd/crash_landing_on_you_has_been_created/',
 'comments': []}

<a id="3"></a>

## 3. Inspect the structure of one thread

I wanted to double-check the structure after I loaded in the data, specifically:

- a `post_id`
- a `title`
- a `selftext`
- a `comments` list

I also need to figure out the best way to deal with the nested reply structure.

In [3]:
# Find a thread that actually has comments so that we can inspect the nesting (it doesn't look like the first one has comments)

example_thread = next(
    thread for thread in threads
    if thread.get("comments")
)

print("Thread ID:", example_thread.get("post_id"))
print("Title:", example_thread.get("title"))
print("Number of top-level comments:", len(example_thread.get("comments", [])))

print("\nFirst top-level comment:")
print(example_thread["comments"][0])

Thread ID: ffh52z
Title: ok, let’s get this started! What are your thoughts about the show?
Number of top-level comments: 10

First top-level comment:
{'comment_id': 'fk0rooq', 'parent_id': 't3_ffh52z', 'author': 'chimichangarolls', 'comment_body': "i'm so emotionally drained and exhausted from being in awe of this show even though i finished watching 2 weeks ago. my brain doesn't work cuz i'm thinking about CLOY and my favorite part of it other than the couple is that the women have really 3D roles... like Seri can cry nonstop but you still know she a boss.", 'score': 19, 'created_utc': 1583764623, 'replies': [{'comment_id': 'fk1c27d', 'parent_id': 't1_fk0rooq', 'author': 'bruisedlee123', 'comment_body': 'Did you have a favorite episode? Mine would have to be ep. 9 ☺️', 'score': 2, 'created_utc': 1583776621, 'replies': [{'comment_id': 'fsfm0lq', 'parent_id': 't1_fk1c27d', 'author': 'tmstms', 'comment_body': "I have not finished it yet, but I already see your point. Episode 9 is when y

<a id="4"></a>

## 4. Flatten the nested comments

We first need a helper function that extracts the text from one comment and all of its descendants.

My first attempt was something like: 

`for comment in thread["comments"]:
    texts.append(comment["comment_body"])`

But that only collects the top-level comments, and the structure is nested. I try to avoid using recursion because I don't like it, but this might be a good time to use it since the depth isn't fixed...

In [4]:
def collect_comment_text(comment):
    """
    Recursively collect the text of one comment and all of its replies.

    Returns:
        list[str]: comment text from this comment and every descendant reply.
    """
    texts = []

    # Add this comment's text.
    body = comment.get("comment_body", "")
    if body and body != "[deleted]":
        texts.append(body)

    # Recursively process every reply.
    for reply in comment.get("replies", []):
        texts.extend(collect_comment_text(reply))

    return texts

Second function time, now that we've flattened out the comments.

Now the whole vector should be the original post title, the body (selftext), every comment, and every nested reply.

In [5]:
def collect_thread_text(thread):
    """
    Collect all text from a complete Reddit thread.
    """
    texts = []

    # Original post title
    title = thread.get("title", "")
    if title and title != "[deleted]":
        texts.append(title)

    # Original post body
    selftext = thread.get("selftext", "")
    if selftext and selftext != "[deleted]":
        texts.append(selftext)

    # All comments and nested replies
    for comment in thread.get("comments", []):
        texts.extend(collect_comment_text(comment))

    return texts

In [6]:
# Test the function on the example thread
example_text_parts = collect_thread_text(example_thread)

print(f"Number of text pieces collected: {len(example_text_parts)}")
print("\nFirst few pieces:\n")

for text in example_text_parts[:5]:
    print(text[:300])
    print("---")

Number of text pieces collected: 24

First few pieces:

ok, let’s get this started! What are your thoughts about the show?
---
I haven’t seen a kdrama for the longest time and it was actually my dad who chose to stream CLOY on Netflix. He stopped watching after the second episode but I went on after immediately falling in love with the leads. Hyun Bin is just so charismatic as Capt. Ri Jeong Hyeok and Son Ye-jin is perfect 
---
i'm so emotionally drained and exhausted from being in awe of this show even though i finished watching 2 weeks ago. my brain doesn't work cuz i'm thinking about CLOY and my favorite part of it other than the couple is that the women have really 3D roles... like Seri can cry nonstop but you still kn
---
Did you have a favorite episode? Mine would have to be ep. 9 ☺️
---
I have not finished it yet, but I already see your point. Episode 9 is when you feel things are going to end happily. Also the stuff in RJH;s parents' home is priceless.
---


The number of collected text pieces should be larger than the number of top-level comments when replies exist, and it does (from 10 to 24), so yay!

<a id="5"></a>

## 5. Turn each thread into one piece of text

For the similarity calculation, each thread needs to become one document, so we join all of the text pieces from the original post and its comments into a single string.

In [7]:
def thread_to_text(thread):
    return " ".join(collect_thread_text(thread))

# Test it on the example thread.
example_full_text = thread_to_text(example_thread)
print(example_full_text[:2_000])

ok, let’s get this started! What are your thoughts about the show? I haven’t seen a kdrama for the longest time and it was actually my dad who chose to stream CLOY on Netflix. He stopped watching after the second episode but I went on after immediately falling in love with the leads. Hyun Bin is just so charismatic as Capt. Ri Jeong Hyeok and Son Ye-jin is perfect as Yoon Se-ri. She’s definitely got the rom-com and drama genre down pat! I also loved Capt. Ri’s squad, most especially Pyo Chi Su ☺️ (we can probably talk about individual characters next time).

CLOY tackled politics, drama and deception, family issues, friendships - and best of all, pure unconditional love, love brought about by fate - all in one highly entertaining and well-written package. 

I’m planning on doing a rewatch soon, when I get a chance! i'm so emotionally drained and exhausted from being in awe of this show even though i finished watching 2 weeks ago. my brain doesn't work cuz i'm thinking about CLOY and my

I'm adding the Reddit ID and title alongside the text, then saving it to a dataframe as the instructions say. THe text itself will turn into the numerical representation used for cosine similarity.

In [8]:
thread_records = []

for thread in threads:
    thread_records.append({
    "thread_id": thread.get("post_id"),
    "title": thread.get("title", ""),
    "text": thread_to_text(thread)
    })

threads_df = pd.DataFrame(thread_records)

print(threads_df.shape)
threads_df.head()

(3075, 3)


,thread_id,title,text
0,dkxewd,Crash Landing On You has been created,Crash Landing On You has been created
1,ffh52z,"ok, let’s get this started! What are your thou...","ok, let’s get this started! What are your thou..."
2,fguorr,Find someone who’ll make you noodles just like...,Find someone who’ll make you noodles just like...
3,fhcrpl,Captain Ri’s squad! Who’s your favorite?,Captain Ri’s squad! Who’s your favorite? I can...
4,fj5byj,Why I think we all need a second season of CLO...,Why I think we all need a second season of CLO...


I'm a little confused as to why the text and the title appear to be the same - we'll see if that causes a problem later on.

<a id="6"></a>

## 6. Tokenize with TextBlob

We just learned how to use TextBlob to tokenize words via `TextBlob(text).words`. Because I'm lazy, I'm making a small helper function so that the same tokenization process can be reused for both versions of the analysis - the first with and the second without stop words.

I'm also normalizing the tokens to lowercase.

I had thoughts on whether to use isalpha() or not - it seemed like a good idea at the time to remove numbers and punctuation, but online text can be kinda hard to standardize. I think it's just important that the same rule is applied across all preprocessing. (I did end up removing, so that there are less tokens and less compute time!)

In [9]:
def textblob_tokens(text):
    """
    Tokenize text with TextBlob and normalize tokens to lowercase.

    Tokens containing no alphabetic characters are removed.
    """
    blob = TextBlob(text)

    return [
        word.lower()
        for word in blob.words
        if word.isalpha()
    ]
    
# Test the tokenizer.
sample_tokens = textblob_tokens(example_full_text)
print(sample_tokens[:50])

['ok', 'let', 's', 'get', 'this', 'started', 'what', 'are', 'your', 'thoughts', 'about', 'the', 'show', 'i', 'haven', 't', 'seen', 'a', 'kdrama', 'for', 'the', 'longest', 'time', 'and', 'it', 'was', 'actually', 'my', 'dad', 'who', 'chose', 'to', 'stream', 'cloy', 'on', 'netflix', 'he', 'stopped', 'watching', 'after', 'the', 'second', 'episode', 'but', 'i', 'went', 'on', 'after', 'immediately', 'falling']


In [10]:
# Tokenize every thread using TextBlob.
threads_df["tokens"] = threads_df["text"].apply(textblob_tokens)

# Check the result.
print(threads_df[["thread_id", "title", "tokens"]].head(3))

  thread_id                                              title  \
0    dkxewd              Crash Landing On You has been created   
1    ffh52z  ok, let’s get this started! What are your thou...   
2    fguorr  Find someone who’ll make you noodles just like...   

                                              tokens  
0      [crash, landing, on, you, has, been, created]  
1  [ok, let, s, get, this, started, what, are, yo...  
2  [find, someone, who, ll, make, you, noodles, j...  


In [11]:
# Count how many times each word occurs in each thread.
word_counts = threads_df["tokens"].apply(Counter)

# Look at the word frequencies for the first thread.
print(word_counts.iloc[0].most_common(20))

[('crash', 1), ('landing', 1), ('on', 1), ('you', 1), ('has', 1), ('been', 1), ('created', 1)]


<a id="7"></a>

## 7. Create word vectors

A thread can contain thousands of words, so we need to represent it numerically.

We're using the bag-of-words representation mentioned in class.

- Each position in the vector represents a word
- Each value represents how many times that word occurs in the thread
- The ordering of the words does not matter, hence the name "bag of words"

I'm constructing these vectors manually, although I assume there's a library somewhere that I could have used instead.

### Creating the shared vocabulary

The vectors for all threads need to have the same length and use the same positions for the same words, so I'm first creating a shared vocabulary containing every unique token that appears in the dataset.

In [12]:
# Create one shared vocabulary containing every unique word.
vocabulary = sorted(set(word for counts in word_counts for word in counts))

print("Number of unique words:", len(vocabulary))
print("First 20 words:", vocabulary[:20])

Number of unique words: 26251
First 20 words: ['a', 'aa', 'aaa', 'aaaa', 'aaaaa', 'aaaaaaa', 'aaaaaaaaaaaaa', 'aaaaaaaaaaaaaaaaa', 'aaaaaaaaaaaaaaaaaaaaaaaaa', 'aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaahhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhh', 'aaaaaaaaaaawwwwwwwwwweee', 'aaaaaaaaaahhhhhhhhhhhyyyyaaaaaayyyyyyyyyy', 'aaaaaah', 'aaaaah', 'aaaaahhhh', 'aaaaat', 'aaaah', 'aaaahhh', 'aaaahhhh', 'aaaand']


### Creating the vectors

For each thread, I now create a vector whose length is the size of the vocabulary.

Each position contains the number of times that word occurs in the thread. A word that does not occur in a particular thread receives a value of 0.

In [13]:
def create_word_vector(word_counts, vocabulary):
    return np.array([
        word_counts.get(word, 0)
        for word in vocabulary
    ])

In [14]:
# Create one numerical vector for every thread.
word_vectors = [
    create_word_vector(counts, vocabulary)
    for counts in word_counts
]

print("Number of vectors:", len(word_vectors))
print("Length of each vector:", len(word_vectors[0]))

# Look at the first 50 entries of the first vector.
word_vectors[0][:50]

Number of vectors: 3075
Length of each vector: 26251


array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0])

The first 50 terms seem to all be 0, but as long as not all values of the vector are 0, we'll be okay.

### Converting the vectors into a matrix

The vectors are currently stored as a list of NumPy arrays, but I'm going to convert them into the term-document matrix we were talking about in class, so that each row represents a thread and each column represents a word in the vocabulary. That way, we can use NumPy's vectorization methods to perform all the similarity calculations quickly instead of looping.

In [15]:
word_matrix = np.array(word_vectors)

print("Matrix shape:", word_matrix.shape)

Matrix shape: (3075, 26251)


<a id="8"></a>

## 8. Calculating Cosine Similarity

The assignment asks me to calculate the similarity between every pair of threads using cosine similarity.

I'm using the cosineSimilarity function provided in class on day 11 to demonstrate the calculation for individual vectors, but I also don't want to let my code run for that long, so I decided to use the dot product of normalized vectors when applying to the whole dataset. (Hence making use of vectorization and not loops)

In [16]:
def cosineSimilarity(vec1, vec2): # the loop way
    """Calculate the cosine similarity between two vectors."""
    V1 = np.array(vec1)
    V2 = np.array(vec2)
    
    denominator = norm(V1) * norm(V2)
    
    if denominator == 0: # I added this to avoid division by zero for an empty vector
        return 0
    
    return np.dot(V1, V2) / denominator

### Normalizing the vectors

For cosine similarity, each vector is divided by its magnitude (done using NumPy).

In [17]:
# Calculate the magnitude of each thread vector.
norms = np.linalg.norm(word_matrix, axis=1, keepdims=True)

# Avoid division by zero for any completely empty thread.
norms[norms == 0] = 1

# Normalize every thread vector.
normalized_matrix = word_matrix / norms

print("Normalized matrix shape:", normalized_matrix.shape)

Normalized matrix shape: (3075, 26251)


### Calculating all pairwise similarities

The dot product of two normalized vectors is their cosine similarity.

Because the normalized vectors are stored as rows of a matrix, multiplying the matrix by its transpose calculates the similarity between every pair of threads. (Yay linear algebra!)

This is much more efficient than calling `cosineSimilarity()` separately inside a Python loop for every pair. (Yay CS345!)

In [18]:
# Calculate cosine similarity between every pair of threads.
similarity_matrix = normalized_matrix @ normalized_matrix.T

print("Similarity matrix shape:", similarity_matrix.shape)

Similarity matrix shape: (3075, 3075)


The resulting matrix is square, where the value at row `i`, column `j` represents the cosine similarity between thread `i` and thread `j`.

Cosine similarity is symmetric, so the similarity between thread A and thread B is the same as the similarity between thread B and thread A. The diagonal contains each thread's similarity with itself, which should be 1 for non-empty vectors.

In [19]:
# Inspect a small portion of the similarity matrix.
similarity_matrix[:5, :5]

array([[1.        , 0.06130797, 0.03314968, 0.01039526, 0.06134623],
       [0.06130797, 1.        , 0.51980987, 0.82754978, 0.80925421],
       [0.03314968, 0.51980987, 1.        , 0.50656116, 0.45742616],
       [0.01039526, 0.82754978, 0.50656116, 1.        , 0.70917588],
       [0.06134623, 0.80925421, 0.45742616, 0.70917588, 1.        ]])

### Checking the calculation

I'm comparing the vectorized calculation against the `cosineSimilarity()` function from class for one pair of threads, just to make sure that the two results are indeed the same (aside from tiny floating-point differences).

In [20]:
# Compare the class function with the vectorized calculation.
test_similarity = cosineSimilarity(word_vectors[0], word_vectors[1])

print("cosineSimilarity function:", test_similarity)
print("Vectorized calculation:", similarity_matrix[0, 1])

cosineSimilarity function: 0.061307967642766754
Vectorized calculation: 0.06130796764276675


It looks like they are, which is good.

<a id="9"></a>

## 9. Creating the Similarity DataFrame

The similarity matrix contains all of the pairwise similarities, as talked about earlier, but the assignment asks me to save the similarities in a dataframe, so I'm going to only include one copy of each pair, excluding the diagonal of the matrix.

In [21]:
thread_ids = threads_df["thread_id"].to_numpy()
titles = threads_df["title"].to_numpy()

pair_rows = []

for i in range(len(threads_df)):
    for j in range(i + 1, len(threads_df)):
        pair_rows.append({
             "thread_id_1": thread_ids[i],
            "title_1": titles[i],
            "thread_id_2": thread_ids[j],
            "title_2": titles[j],
            "similarity": similarity_matrix[i, j]
        })

similarity_df = pd.DataFrame(pair_rows)

print("Number of unique thread pairs:", len(similarity_df))
similarity_df.head()

Number of unique thread pairs: 4726275


,thread_id_1,title_1,thread_id_2,title_2,similarity
0,dkxewd,Crash Landing On You has been created,ffh52z,"ok, let’s get this started! What are your thou...",0.061308
1,dkxewd,Crash Landing On You has been created,fguorr,Find someone who’ll make you noodles just like...,0.033150
2,dkxewd,Crash Landing On You has been created,fhcrpl,Captain Ri’s squad! Who’s your favorite?,0.010395
3,dkxewd,Crash Landing On You has been created,fj5byj,Why I think we all need a second season of CLO...,0.061346
4,dkxewd,Crash Landing On You has been created,fjimic,Yoon Seri's Fashion - I bought the Gentle Mons...,0.024708


In [22]:
OUTPUT_PATH = Path("data/cs315-week4-data/CLOY/cloy_thread_similarity_with_stopwords.csv")

similarity_df.to_csv(OUTPUT_PATH, index=False)

print(f"Saved similarity data to: {OUTPUT_PATH}")

Saved similarity data to: data/cs315-week4-data/CLOY/cloy_thread_similarity_with_stopwords.csv


<a id="11"></a>

## 11. Find the five most similar threads

The assignment asks for a function that takes one thread ID and prints the original thread ID and title and the five most similar thread IDs and titles

We can do this directly from the pairwise dataframe, but because of the way the dataframe is set up, the requested thread may appear in either thread_id_1 or thread_id_2, so we check both columns.

In [23]:
def top_5_similar(thread_id, pairwise_df=similarity_df):
    """
    Print the given thread ID/title and its five most similar threads.
    """
    # Check that the thread exists.
    matches = threads_df[threads_df["thread_id"] == thread_id]

    if matches.empty:
        raise ValueError(f"Thread ID {thread_id!r} was not found.")

    original = matches.iloc[0]

    print(f"Original thread: {original['thread_id']}")
    print(f"Title: {original['title']}")
    print("\nTop 5 most similar threads:")

    # Find every pair involving this thread.
    relevant = pairwise_df[
        (pairwise_df["thread_id_1"] == thread_id) |
        (pairwise_df["thread_id_2"] == thread_id)
    ].copy()

    # Identify the other thread in each pair.
    relevant["other_thread_id"] = np.where(
        relevant["thread_id_1"] == thread_id,
        relevant["thread_id_2"],
        relevant["thread_id_1"]
    )

    relevant["other_title"] = np.where(
        relevant["thread_id_1"] == thread_id,
        relevant["title_2"],
        relevant["title_1"]
    )

    top5 = relevant.sort_values("similarity", ascending=False).head(5)

    for _, row in top5.iterrows():
        print(
            f"{row['other_thread_id']} | "
            f"{row['similarity']:.4f} | "
            f"{row['other_title']}"
        )

    return top5[[
        "other_thread_id",
        "other_title",
        "similarity"
    ]].reset_index(drop=True)

And then test with an example.

In [24]:
example_thread_id = threads_df.iloc[0]["thread_id"]

top_5_similar(example_thread_id)

Original thread: dkxewd
Title: Crash Landing On You has been created

Top 5 most similar threads:
zoafq7 | 0.6172 | [FULL LYRIC] CRASH LANDING ON YOU
1ntfysd | 0.5714 | CRASH LANDING ON YOU PLOT HOLES
umtgh8 | 0.5714 | 50 Best Quotes from "Crash Landing On You"
fqh4pi | 0.5669 | Crash Landing on You Season 2 Details!
g6oy82 | 0.5345 | My Crash Landing on You Penguin art!! :D


,other_thread_id,other_title,similarity
0,zoafq7,[FULL LYRIC] CRASH LANDING ON YOU,0.617213
1,1ntfysd,CRASH LANDING ON YOU PLOT HOLES,0.571429
2,umtgh8,"50 Best Quotes from ""Crash Landing On You""",0.571429
3,fqh4pi,Crash Landing on You Season 2 Details!,0.566947
4,g6oy82,My Crash Landing on You Penguin art!! :D,0.534522


<a id="11"></a>

## 11. Verification

I used the thread_explorer app to inspect the five threads with the highest cosine similarity. I would say the results are pretty similar, enough to make sense, because the threads shared vocabulary and discussed overlapping topics about CLOY, but shorter titles are definitely hard because using the phrase "Crash Landing on You" guarantees at least 4 overlapping tokens. Because of this, two threads can receive a high score because they use many of the same words without expressing exactly the same idea.

<a id="12"></a>

## 12. Stop words: repeat the process

Time to do the same thing thing but without stop words such as the, a, and, is, etc. 

First, let's inspect the list.

I had originally tried to get the length of stopwords, but I think it's a lazy loader, so I changed it to stopwords.words().

In [25]:
from nltk.corpus import stopwords

# Fetch the actual list of English stop words
stop_words_list = stopwords.words('english')

print("Number of stop words:", len(stop_words_list))
print(stop_words_list[:50])

Number of stop words: 198
['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn']


I now modify the tokenizer so that it removes any token that appears in TextBlob's English stop-word list, while keeping .lower() and .isalpha().

In [26]:
def textblob_tokens_no_stopwords(text):
    """
    Tokenize text with TextBlob, normalize tokens to lowercase,
    and remove English stop words.
    """
    blob = TextBlob(text)

    return [
        word.lower()
        for word in blob.words
        if word.lower() not in stop_words_list and word.isalpha()
    ]

In [27]:
# Test the tokenizer without stop words.
sample_tokens_no_stopwords = textblob_tokens_no_stopwords(example_full_text)

print(sample_tokens_no_stopwords[:50])

['ok', 'let', 'get', 'started', 'thoughts', 'show', 'seen', 'kdrama', 'longest', 'time', 'actually', 'dad', 'chose', 'stream', 'cloy', 'netflix', 'stopped', 'watching', 'second', 'episode', 'went', 'immediately', 'falling', 'love', 'leads', 'hyun', 'bin', 'charismatic', 'capt', 'ri', 'jeong', 'hyeok', 'son', 'perfect', 'yoon', 'definitely', 'got', 'drama', 'genre', 'pat', 'also', 'loved', 'capt', 'ri', 'squad', 'especially', 'pyo', 'chi', 'su', 'probably']


Then I tokenize every thread again, without stop words, and add it as a separate column to the df (so that I can compare both versions later).

In [28]:
# Tokenize every thread while removing stop words.
threads_df["tokens_no_stopwords"] = threads_df["text"].apply(
    textblob_tokens_no_stopwords
)

# Check the result.
print(
    threads_df[
        ["thread_id", "title", "tokens_no_stopwords"]
    ].head(3)
)

  thread_id                                              title  \
0    dkxewd              Crash Landing On You has been created   
1    ffh52z  ok, let’s get this started! What are your thou...   
2    fguorr  Find someone who’ll make you noodles just like...   

                                 tokens_no_stopwords  
0                          [crash, landing, created]  
1  [ok, let, get, started, thoughts, show, seen, ...  
2  [find, someone, make, noodles, like, ri, jeong...  


Next, I count the frequency of each remaining word in each thread.

In [29]:
# Count how many times each remaining word occurs in each thread.
word_counts_no_stopwords = threads_df["tokens_no_stopwords"].apply(Counter)

# Look at the word frequencies for the first thread.
print(word_counts_no_stopwords.iloc[0].most_common(20))

[('crash', 1), ('landing', 1), ('created', 1)]


I'm redoing the shared vocabulary because the no-stopword vocab is now smaller.

In [30]:
# Create a shared vocabulary without stop words.
vocabulary_no_stopwords = sorted(
    set(
        word
        for counts in word_counts_no_stopwords
        for word in counts
    )
)

print("Number of unique words:", len(vocabulary_no_stopwords))
print("First 20 words:", vocabulary_no_stopwords[:20])

Number of unique words: 26102
First 20 words: ['aa', 'aaa', 'aaaa', 'aaaaa', 'aaaaaaa', 'aaaaaaaaaaaaa', 'aaaaaaaaaaaaaaaaa', 'aaaaaaaaaaaaaaaaaaaaaaaaa', 'aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaahhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhh', 'aaaaaaaaaaawwwwwwwwwweee', 'aaaaaaaaaahhhhhhhhhhhyyyyaaaaaayyyyyyyyyy', 'aaaaaah', 'aaaaah', 'aaaaahhhh', 'aaaaat', 'aaaah', 'aaaahhh', 'aaaahhhh', 'aaaand', 'aaaargh']


Then I need to create the vectors again, but I can reuse the `create_word_vector()` function from earlier, as long as I give it the updated no-stopword shared vocabulary.

In [31]:
# Create one numerical vector for every thread.
word_vectors_no_stopwords = [
    create_word_vector(counts, vocabulary_no_stopwords)
    for counts in word_counts_no_stopwords
]

print("Number of vectors:", len(word_vectors_no_stopwords))
print("Length of each vector:", len(word_vectors_no_stopwords[0]))

# Look at the first 50 entries of the first vector.
word_vectors_no_stopwords[0][:50]

Number of vectors: 3075
Length of each vector: 26102


array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0])

It looks like the vocabulary is marginally smaller, by around 150 words?

Next is to combine the vectors into a matrix, just like before.

In [32]:
word_matrix_no_stopwords = np.array(word_vectors_no_stopwords)

print("Matrix shape:", word_matrix_no_stopwords.shape)

Matrix shape: (3075, 26102)


And then cosine similarity.

In [33]:
# Calculate the magnitude of each thread vector.
norms_no_stopwords = np.linalg.norm(
    word_matrix_no_stopwords,
    axis=1,
    keepdims=True
)

# Avoid division by zero for completely empty threads.
norms_no_stopwords[norms_no_stopwords == 0] = 1

# Normalize every thread vector.
normalized_matrix_no_stopwords = (
    word_matrix_no_stopwords / norms_no_stopwords
)

print(
    "Normalized matrix shape:",
    normalized_matrix_no_stopwords.shape
)

Normalized matrix shape: (3075, 26102)


In [34]:
# Calculate cosine similarity between every pair of threads.
similarity_matrix_no_stopwords = (
    normalized_matrix_no_stopwords
    @ normalized_matrix_no_stopwords.T
)

print(
    "Similarity matrix shape:",
    similarity_matrix_no_stopwords.shape
)

Similarity matrix shape: (3075, 3075)


As before, the resulting matrix is symmetric, with each entry representing the cosine similarity between two threads, and the mini-inspection looks right.

In [35]:
# Inspect a small portion of the similarity matrix.
similarity_matrix_no_stopwords[:5, :5]

array([[1.        , 0.        , 0.        , 0.        , 0.02963695],
       [0.        , 1.        , 0.13119876, 0.36023638, 0.40660832],
       [0.        , 0.13119876, 1.        , 0.19360917, 0.08150161],
       [0.        , 0.36023638, 0.19360917, 1.        , 0.21282708],
       [0.02963695, 0.40660832, 0.08150161, 0.21282708, 1.        ]])

Then I can save it to as a CSV.

In [36]:
pair_rows_no_stopwords = []

for i in range(len(threads_df)):
    for j in range(i + 1, len(threads_df)):
        pair_rows_no_stopwords.append({
            "thread_id_1": thread_ids[i],
            "title_1": titles[i],
            "thread_id_2": thread_ids[j],
            "title_2": titles[j],
            "similarity": similarity_matrix_no_stopwords[i, j]
        })

similarity_df_no_stopwords = pd.DataFrame(pair_rows_no_stopwords)

print(
    "Number of unique thread pairs:",
    len(similarity_df_no_stopwords)
)

similarity_df_no_stopwords.head()

Number of unique thread pairs: 4726275


,thread_id_1,title_1,thread_id_2,title_2,similarity
0,dkxewd,Crash Landing On You has been created,ffh52z,"ok, let’s get this started! What are your thou...",0.000000
1,dkxewd,Crash Landing On You has been created,fguorr,Find someone who’ll make you noodles just like...,0.000000
2,dkxewd,Crash Landing On You has been created,fhcrpl,Captain Ri’s squad! Who’s your favorite?,0.000000
3,dkxewd,Crash Landing On You has been created,fj5byj,Why I think we all need a second season of CLO...,0.029637
4,dkxewd,Crash Landing On You has been created,fjimic,Yoon Seri's Fashion - I bought the Gentle Mons...,0.000000


In [37]:
OUTPUT_PATH_NO_STOPWORDS = Path("data/cs315-week4-data/CLOY/cloy_thread_similarity_without_stopwords.csv")

similarity_df_no_stopwords.to_csv(
    OUTPUT_PATH_NO_STOPWORDS,
    index=False
)

print(
    f"Saved similarity data to: {OUTPUT_PATH_NO_STOPWORDS}"
)

Saved similarity data to: data/cs315-week4-data/CLOY/cloy_thread_similarity_without_stopwords.csv


### Top 5 Analysis

I'm using a different function so that the original values don't change. (Although I probably could have just used the same function and added parameters.)

In [38]:
def top_5_similar_no_stopwords(
    thread_id,
    pairwise_df=similarity_df_no_stopwords
):
    """
    Print the given thread ID/title and its five most similar
    threads after removing stop words.
    """
    # Check that the thread exists.
    matches = threads_df[threads_df["thread_id"] == thread_id]

    if matches.empty:
        raise ValueError(f"Thread ID {thread_id!r} was not found.")

    original = matches.iloc[0]

    print(f"Original thread: {original['thread_id']}")
    print(f"Title: {original['title']}")
    print("\nTop 5 most similar threads without stop words:")

    # Find every pair involving this thread.
    relevant = pairwise_df[
        (pairwise_df["thread_id_1"] == thread_id) |
        (pairwise_df["thread_id_2"] == thread_id)
    ].copy()

    # Identify the other thread in each pair.
    relevant["other_thread_id"] = np.where(
        relevant["thread_id_1"] == thread_id,
        relevant["thread_id_2"],
        relevant["thread_id_1"]
    )

    relevant["other_title"] = np.where(
        relevant["thread_id_1"] == thread_id,
        relevant["title_2"],
        relevant["title_1"]
    )

    top5 = relevant.sort_values(
        "similarity",
        ascending=False
    ).head(5)

    for _, row in top5.iterrows():
        print(
            f"{row['other_thread_id']} | "
            f"{row['similarity']:.4f} | "
            f"{row['other_title']}"
        )

    return top5[[
        "other_thread_id",
        "other_title",
        "similarity"
    ]].reset_index(drop=True)

And then I'm testing it with the same thread so I can compare.

In [39]:
top_5_similar_no_stopwords(example_thread_id)

Original thread: dkxewd
Title: Crash Landing On You has been created

Top 5 most similar threads without stop words:
g6oy82 | 0.5774 | My Crash Landing on You Penguin art!! :D
umtgh8 | 0.5774 | 50 Best Quotes from "Crash Landing On You"
zoafq7 | 0.5774 | [FULL LYRIC] CRASH LANDING ON YOU
1n7k3ec | 0.5164 | Crash Landing On my Heart
hdejht | 0.5164 | avengers portal scene but it's crash landing on you


,other_thread_id,other_title,similarity
0,g6oy82,My Crash Landing on You Penguin art!! :D,0.577350
1,umtgh8,"50 Best Quotes from ""Crash Landing On You""",0.577350
2,zoafq7,[FULL LYRIC] CRASH LANDING ON YOU,0.577350
3,1n7k3ec,Crash Landing On my Heart,0.516398
4,hdejht,avengers portal scene but it's crash landing o...,0.516398


<a id="13"></a>

## 13. Compare the two approaches

I guess I should look at things like:
 - Did the five nearest threads stay the same?
 - Did their ordering change?
 - Did the numerical similarity scores change?
 - Did removing stop words make the results more focused on content words?

In [40]:
# Compare the top five before and after removing stop words.

print("WITH STOP WORDS")
top_with_stopwords = top_5_similar(
example_thread_id,
similarity_df
)

print("\n" + "=" * 80 + "\n")

print("WITHOUT STOP WORDS")
top_without_stopwords = top_5_similar_no_stopwords(
example_thread_id,
similarity_df_no_stopwords
)

WITH STOP WORDS
Original thread: dkxewd
Title: Crash Landing On You has been created

Top 5 most similar threads:
zoafq7 | 0.6172 | [FULL LYRIC] CRASH LANDING ON YOU
1ntfysd | 0.5714 | CRASH LANDING ON YOU PLOT HOLES
umtgh8 | 0.5714 | 50 Best Quotes from "Crash Landing On You"
fqh4pi | 0.5669 | Crash Landing on You Season 2 Details!
g6oy82 | 0.5345 | My Crash Landing on You Penguin art!! :D


WITHOUT STOP WORDS
Original thread: dkxewd
Title: Crash Landing On You has been created

Top 5 most similar threads without stop words:
g6oy82 | 0.5774 | My Crash Landing on You Penguin art!! :D
umtgh8 | 0.5774 | 50 Best Quotes from "Crash Landing On You"
zoafq7 | 0.5774 | [FULL LYRIC] CRASH LANDING ON YOU
1n7k3ec | 0.5164 | Crash Landing On my Heart
hdejht | 0.5164 | avengers portal scene but it's crash landing on you


In [41]:
ids_with_stopwords = set(top_with_stopwords["other_thread_id"])
ids_without_stopwords = set(top_without_stopwords["other_thread_id"])

overlap = ids_with_stopwords & ids_without_stopwords

print("Top-5 threads with stop words:", ids_with_stopwords)
print("Top-5 threads without stop words:", ids_without_stopwords)
print("Number of threads appearing in both top-5 lists:", len(overlap))

Top-5 threads with stop words: {'umtgh8', '1ntfysd', 'g6oy82', 'fqh4pi', 'zoafq7'}
Top-5 threads without stop words: {'umtgh8', '1n7k3ec', 'g6oy82', 'zoafq7', 'hdejht'}
Number of threads appearing in both top-5 lists: 3


It looks like there's a decent amount of overlap (3/5 threads), but the rankings did change, and two new threads were introduced.